# Extração de frames de falso positivo

Autor:  Viviane da Rosa Sommer

Entrega: 20/03/2025

## Objetivo:
Notebook para exxtrair os frames de falso positivo dos vídeos de Jaqueta.

## Importações Necessárias

In [ ]:
!pip install imagehash
!pip install pillow
!pip install pandas
!pip install opencv-python-headless

In [ ]:
import os
import pandas as pd
import cv2
import imagehash
from PIL import Image

## Declaração de Constantes e Modelos

In [ ]:
csv_folder = "Videos-Resultado/CSV"
video_folder = "Videos"
output_folder = "Dataset-Jaqueta-Negativo-3"

## Funções Necessárias

In [ ]:
def extract_frames_from_video(csv_folder: str, video_folder: str, output_folder: str, score_threshold: float = 0.8) -> None:
    """
    Processes CSV files and extracts frames from videos based on specified conditions.

    Args:
        csv_folder (str): Path to the folder containing CSV files.
        video_folder (str): Path to the folder containing videos corresponding to the CSVs.
        output_folder (str): Path to the folder where the extracted frames will be saved.
        score_threshold (float): Minimum 'Average Score' value to extract frames.

    Returns:
        None
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    processed_hashes = set()

    for csv_file in os.listdir(csv_folder):
        if csv_file.endswith(".csv"):
            csv_path = os.path.join(csv_folder, csv_file)
            df = pd.read_csv(csv_path)

            filtered_frames = df[(df['Average Score'] > 0.50) & (df['Average Score'] < score_threshold)]

            video_name = os.path.splitext(csv_file)[0] + '.mp4'
            video_path = os.path.join(video_folder, video_name)

            if not os.path.exists(video_path):
                print(f"Corresponding video {video_name} not found. Skipping...")
                continue

            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened():
                print(f"Error opening video {video_path}. Skipping...")
                continue

            fps = int(cap.get(cv2.CAP_PROP_FPS))

            for _, row in filtered_frames.iterrows():
                time_in_seconds = int(row['Time (seconds)'])
                frame_number = time_in_seconds * fps

                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
                ret, frame = cap.read()

                if not ret:
                    print(f"Unable to read frame {frame_number} from video {video_name}. Skipping...")
                    continue

                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                pil_image = Image.fromarray(frame_rgb)

                hash_value = imagehash.average_hash(pil_image)

                if hash_value not in processed_hashes:
                    processed_hashes.add(hash_value)

                    output_filename = f"{os.path.splitext(csv_file)[0]}_frame_{time_in_seconds}.png"
                    output_path = os.path.join(output_folder, output_filename)
                    pil_image.save(output_path)
                    print(f"Extracted frame saved: {output_path}")

            cap.release()

    print("Processing completed!")

## Extração dos frames

In [ ]:
extract_frames_from_video(csv_folder, video_folder, output_folder)